# 3. Data Preparation
## Feature Engineering & Matrix Construction for CRISP-DM
In this notebook, we apply robust pipeline strategies to cleanly move our relational dataset from raw tables to a tightly encoded, mathematically scaled feature matrix. This ensures our downstream Principal Component Analysis (PCA) and K-Means segmentation algorithms execute without mathematical biases.


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
print("Libraries Loaded.")


### 3.1 Bronze Layer: Ingest Raw Data
Loading the raw tables identically to our EDA process.


In [ ]:
data_dir = 'data'
customers_df = pd.read_csv(os.path.join(data_dir, 'Customers.csv'))
products_df = pd.read_csv(os.path.join(data_dir, 'Products.csv'))
transactions_df = pd.read_csv(os.path.join(data_dir, 'Transactions.csv'))

print(f"Bronze Tables Loaded - Trxn: {len(transactions_df)}")


### 3.2 Silver Layer: Denormalization & Contract Validation
We merge the datasets and explicitly drop chronological metrics (adhering to project constraints). We explicitly check for NULL proliferation from bad joins.


In [ ]:
# Merge into Silver Master Dataset
silver_df = transactions_df.merge(customers_df, on='CustomerID', how='left')
silver_df = silver_df.merge(products_df, on='ProductID', how='left')

# Drop Temporal Variables
temporal_cols = ['TransactionDate', 'SignupDate']
silver_df = silver_df.drop(columns=[col for col in temporal_cols if col in silver_df.columns])

# Implicit Contract Validation: No Nulls Allowed
assert silver_df.isnull().sum().sum() == 0, "Data Contract Violation: Nulls found in the Silver Layer."
print("Silver Layer built and validated. Rows:", len(silver_df))


### 3.3 Gold Layer: Customer Behavioral Aggregation (FM Analysis)
K-Means clusters users, not individual transactions. We aggregate down to the `CustomerID` to construct behavioral profiles.


In [ ]:
def get_mode(series):
    return series.mode()[0] if not series.empty else None

# Aggregate to Gold Customer metrics
gold_df = silver_df.groupby('CustomerID').agg(
    Frequency=('TransactionID', 'count'),
    Total_Expenditure=('TotalValue', 'sum'),
    Average_Order_Value=('TotalValue', 'mean'),
    Total_Items_Bought=('Quantity', 'sum'),
    Region=('Region', get_mode),
    Preferred_Category=('Category', get_mode)
).reset_index()

print("Gold Layer built. Customer Profiles:", len(gold_df))
gold_df.head()


### 3.4 Matrix Construction: Categorical One-Hot Encoding
PCA and K-Means understand only vectors. We convert text-based categories (`Region`, `Preferred_Category`) into numerical binary matrices.


In [ ]:
# Set CustomerID as Index so it is preserved but ignored mathematically
gold_df.set_index('CustomerID', inplace=True)

# Generate One-Hot Encoded variables
encoded_df = pd.get_dummies(gold_df, columns=['Region', 'Preferred_Category'], drop_first=False)

# Convert boolean matrices to integers (1, 0)
for col in encoded_df.columns:
    if encoded_df[col].dtype == 'bool':
        encoded_df[col] = encoded_df[col].astype(int)

print("Categoricals Encoded. Shape:", encoded_df.shape)
encoded_df.head()


### 3.5 Matrix Construction: Feature Scaling
**Critical Step:** If we do not scale numerical values, `Total_Expenditure` (in thousands) will completely dominate PCA's variance outputs, rendering `Frequency` (single digits) useless. `StandardScaler` standardizes features by removing the mean and scaling to unit variance.


In [ ]:
scaler = StandardScaler()

# Identify features to scale
features = encoded_df.columns

# Scale features
scaled_arr = scaler.fit_transform(encoded_df)
scaled_df = pd.DataFrame(scaled_arr, columns=features, index=encoded_df.index)

print("Data Standardized (Mean: ~0, Std: ~1). Ready for PCA.")
scaled_df.head()


### 3.6 Data Pipeline Export
We save the fully prepared feature matrix to a CSV. This acts as the single source of truth for the upcoming `pca.ipynb` and machine learning tasks.


In [ ]:
# Save analytical dataset for PCA
export_path = os.path.join(data_dir, 'customer_features_scaled.csv')
scaled_df.to_csv(export_path)

print(f"Feature matrix exported securely to: {export_path}")
